# MARBERT Fine-tuning — Saudi Arabic NLP (Intent + Sentiment)

**Goal:** answer one question — *does a transformer (MARBERT) add real value over TF-IDF + LR/SVM on our 308-row dataset?*

**Before you run:**
1. `Runtime > Change runtime type > T4 GPU`
2. Have these files ready to upload (from the baseline notebook, so the split is IDENTICAL): `train.csv`, `val.csv`, `test.csv`
3. Optional, for the final comparison table: `baseline_results.csv`

**Protocol (kept identical to the baselines):** same Train/Val/Test split, same `text_clean` column, same class-balanced loss idea (like `class_weight="balanced"`).
The best epoch is picked on **Val**, and **Test** is used once per run. We repeat with 3 seeds because transformers on tiny data are unstable.

In [1]:
!pip install -q transformers

In [2]:
import os, random, json, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cpu":
    print("WARNING: no GPU detected. Runtime > Change runtime type > T4 GPU (training on CPU will be very slow).")

# ---------------- Config (change here) ----------------
MODEL_NAME = "UBC-NLP/MARBERT"   # alternative to try later: "UBC-NLP/MARBERTv2"
TEXT_COL   = "text_clean"        # same column the baselines used (fair comparison)
MAX_LEN    = 64
BATCH_SIZE = 16
EPOCHS     = 10
LR         = 2e-5
SEEDS      = [42, 43, 44]
TASKS      = ["intent", "sentiment"]

device: cuda


In [3]:
# Load the SAME split used by the baseline notebook
if not all(os.path.exists(f) for f in ["train.csv", "val.csv", "test.csv"]):
    from google.colab import files
    print("Upload train.csv, val.csv, test.csv (and optionally baseline_results.csv)")
    files.upload()

train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("val.csv")
test_df  = pd.read_csv("test.csv")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
assert (len(train_df), len(val_df), len(test_df)) == (246, 31, 31), "Split sizes differ from the baseline split!"

Train: 246 | Val: 31 | Test: 31


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Sanity check: is MAX_LEN long enough for our messages?
tok_lens = [len(tokenizer.encode(t)) for t in train_df[TEXT_COL].astype(str)]
print(f"Token length -> mean: {np.mean(tok_lens):.1f} | max: {max(tok_lens)} | MAX_LEN: {MAX_LEN}")
print("Example tokens:", tokenizer.tokenize(str(train_df[TEXT_COL].iloc[0])))

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Token length -> mean: 9.6 | max: 18 | MAX_LEN: 64
Example tokens: ['ابغى', 'نسخة', 'ورقية', 'من', 'العقد', 'مو', 'بس', 'الكترونية']


In [5]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def encode(df, label2id, task):
    enc = tokenizer(df[TEXT_COL].astype(str).tolist(), padding="max_length",
                    truncation=True, max_length=MAX_LEN, return_tensors="pt")
    labels = torch.tensor(df[task].map(label2id).values, dtype=torch.long)
    return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)

def predict(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for ids, mask, _ in loader:
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            preds.extend(logits.argmax(dim=-1).cpu().tolist())
    return np.array(preds)

def metrics(y_true, y_pred):
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    _, _, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred), "precision_macro": p_m,
            "recall_macro": r_m, "f1_macro": f1_m, "f1_weighted": f1_w}

In [6]:
def run_one(task, seed):
    set_seed(seed)
    labels = sorted(train_df[task].unique())
    label2id = {l: i for i, l in enumerate(labels)}

    train_ds = encode(train_df, label2id, task)
    val_ds   = encode(val_df, label2id, task)
    test_ds  = encode(test_df, label2id, task)
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g)
    val_loader   = DataLoader(val_ds, batch_size=64)
    test_loader  = DataLoader(test_ds, batch_size=64)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(labels)).to(device)

    # Class-balanced loss (same idea as class_weight="balanced" in LR / SVM)
    y_tr = train_df[task].map(label2id).values
    counts = np.bincount(y_tr, minlength=len(labels))
    weights = torch.tensor(len(y_tr) / (len(labels) * counts), dtype=torch.float).to(device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

    y_val  = val_df[task].map(label2id).values
    y_test = test_df[task].map(label2id).values

    best_f1, best_epoch, best_state = -1.0, -1, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for ids, mask, y in train_loader:
            optimizer.zero_grad()
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            loss = loss_fn(logits, y.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        val_f1 = metrics(y_val, predict(model, val_loader))["f1_macro"]
        if val_f1 > best_f1:   # model selection on VAL only
            best_f1, best_epoch = val_f1, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    val_m  = metrics(y_val, predict(model, val_loader))
    test_pred = predict(model, test_loader)
    test_m = metrics(y_test, test_pred)

    row = {"task": task, "seed": seed, "best_epoch": best_epoch,
           **{f"val_{k}": v for k, v in val_m.items()},
           **{f"test_{k}": v for k, v in test_m.items()}}
    preds = {"y_test": [labels[i] for i in y_test], "y_pred": [labels[i] for i in test_pred]}

    del model, optimizer, best_state
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return row, preds

In [7]:
# Runs 2 tasks x 3 seeds = 6 fine-tunings (a few minutes on a T4)
all_rows, all_preds = [], {}
for task in TASKS:
    for seed in SEEDS:
        t0 = time.time()
        row, preds = run_one(task, seed)
        all_rows.append(row)
        all_preds[f"{task}__MARBERT__seed{seed}"] = preds
        print(f"[{task:9s}] seed={seed} best_epoch={row['best_epoch']:2d} | "
              f"val F1={row['val_f1_macro']:.3f} | "
              f"TEST acc={row['test_accuracy']:.3f} F1(macro)={row['test_f1_macro']:.3f} "
              f"({time.time() - t0:.0f}s)")

res = pd.DataFrame(all_rows)
res.to_csv("marbert_results.csv", index=False)
with open("marbert_test_predictions.json", "w", encoding="utf-8") as f:
    json.dump(all_preds, f, ensure_ascii=False, indent=2)
print("\nSaved: marbert_results.csv, marbert_test_predictions.json")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  654MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider

[sentiment] seed=43 best_epoch= 5 | val F1=1.000 | TEST acc=0.935 F1(macro)=0.930 (39s)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider

[sentiment] seed=44 best_epoch= 5 | val F1=1.000 | TEST acc=0.903 F1(macro)=0.882 (39s)

Saved: marbert_results.csv, marbert_test_predictions.json


In [4]:
# Mean +- std over seeds (test set)
summary = res.groupby("task")[["test_accuracy", "test_f1_macro", "test_f1_weighted"]].agg(["mean", "std"]).round(3)
print(summary)

          test_accuracy        test_f1_macro        test_f1_weighted       
                   mean    std          mean    std             mean    std
task                                                                       
intent            0.914  0.037         0.930  0.030            0.916  0.035
sentiment         0.914  0.019         0.898  0.028            0.911  0.021


In [5]:
# Final comparison table: baselines (test split) vs MARBERT (mean over seeds)
rows = []
if os.path.exists("baseline_results.csv"):
    b = pd.read_csv("baseline_results.csv")
    b = b[b["split"] == "test"]
    for _, r in b.iterrows():
        rows.append({"task": r["task"], "model": r["model"], "accuracy": r["accuracy"],
                     "f1_macro": r["f1_macro"], "f1_std_over_seeds": np.nan})
else:
    print("baseline_results.csv not found -> upload it to include LR/SVM in this table.")

for task, g in res.groupby("task"):
    rows.append({"task": task, "model": f"MARBERT (mean of {len(g)} seeds)",
                 "accuracy": g["test_accuracy"].mean(), "f1_macro": g["test_f1_macro"].mean(),
                 "f1_std_over_seeds": g["test_f1_macro"].std()})

comp = pd.DataFrame(rows).sort_values("task", kind="stable").reset_index(drop=True)
comp.to_csv("final_comparison.csv", index=False)
print(comp.round(3).to_string(index=False))

baseline_results.csv not found -> upload it to include LR/SVM in this table.
     task                     model  accuracy  f1_macro  f1_std_over_seeds
   intent MARBERT (mean of 3 seeds)     0.914     0.930              0.030
sentiment MARBERT (mean of 3 seeds)     0.914     0.898              0.028


### How to read the result (be careful!)
- The Test set is only **31 rows** -> one prediction = ~3.2 points. A gap of a few points between MARBERT and LR/SVM is **noise**.
- Look at the **std across seeds** (`f1_std_over_seeds`). If MARBERT's gain is smaller than that spread, treat it as "no clear difference".
- A real signal looks like: MARBERT beats the baselines by a clear margin (roughly 5+ points F1-macro) **and** consistently on all 3 seeds.
- Also compare against the **cross-validation** numbers from the baseline notebook (Intent ~0.73-0.74, Sentiment ~0.80-0.83), which are more stable than the 31-row test.
- Next step: confusion matrices / error analysis using `marbert_test_predictions.json` and `test_predictions.json`.